# Evidências sobre o desempenho da EWMA3

Este notebook investiga por que os modelos têm dificuldade para melhorar a EWMA3. Ele reúne evidências fora da amostra sobre: desempenho por alvo e fold; persistência mês a mês dentro de cada H3 (removendo o nível médio da célula); previsão dos erros da EWMA3; efeito da divisão alto/baixo; concentração de ocorrências no grupo alto; e importância das features para o resíduo.

Os folds são jan–jun/2024, jul–dez/2024 e jan–jun/2025. 2022 pode fornecer histórico anterior, mas não é período de treino. Jul/2025–jun/2026 segue reservado. A validação existente é rolling one-step-ahead: observações mensais anteriores dentro do período de validação alimentam o histórico para o mês seguinte.

O notebook separa **evidência** (medidas calculadas nos dados) de **interpretação** (hipóteses sobre por que ocorre). Correlação e importância preditiva não demonstram causalidade. Com três folds, diferenças pequenas devem ser tratadas como frágeis.

Execute a célula para gerar tabelas e CSVs com prefixo `evidence_why_ewm3_h3_` em `datasets/`.

## Evidências já calculadas

| Alvo | RMSE média histórica fixa | RMSE EWMA3 | Redução de RMSE da EWMA3 | Correlação intra-H3: atual vs lag 1 | Correlação intra-H3: atual vs EWMA3 |
|---|---:|---:|---:|---:|---:|
| CRIME | 0,02234 | 0,02036 | 8,82% | 0,33 | 0,40 |
| FURTO | 0,02422 | 0,02327 | 3,95% | 0,24 | 0,29 |
| ROUBO | 0,03533 | 0,03038 | 13,74% | 0,43 | 0,48 |

O grupo alto (20% dos H3) concentrou em média **90% de CRIME, 92% de FURTO e 89% de ROUBO** no treino. A divisão por grupos, sozinha, não garantiu superar EWM3.

Na comparação ampliada, modelos que preveem o resíduo do EWM3 foram avaliados globalmente e com separação alto/baixo. Os melhores resíduos globais foram RF para CRIME (ganho médio de 1,47%, vencendo 2/3 folds), RF para FURTO (0,19%, 2/3 folds) e GAM para ROUBO (3,35%, 3/3 folds). O ganho para FURTO é muito pequeno; mesmo para os outros alvos, três folds ainda são evidência limitada. A tabela atualizada completa aparece no dashboard e no CSV de comparação.

In [1]:
from pathlib import Path
import runpy
script = Path('processing_datasets/evidence_why_ewm3_strong_h3.py')
if not script.exists():
    script = Path('evidence_why_ewm3_strong_h3.py')
runpy.run_path(str(script), run_name='__main__')

1) Baselines e melhores algoritmos por escopo:
     target                      escopo                          modelo  RMSE_medio  ganho_medio_vs_EWM3_pct  folds_melhor_EWM3  ganho_medio_vs_media_fixa_pct
RISCO_CRIME    alto_modelo_baixo_hurdle        EWM3_alto_HurdleNB_baixo    0.020412                -0.244584                  1                       8.598583
RISCO_CRIME    alto_modelo_baixo_hurdle Ridge_log1p_alto_HurdleNB_baixo    0.020959                -3.026900                  1                       6.041862
RISCO_CRIME    alto_modelo_baixo_hurdle         GAM_alto_HurdleNB_baixo    0.021254                -4.540091                  1                       4.653643
RISCO_CRIME    alto_modelo_baixo_hurdle          RF_alto_HurdleNB_baixo    0.021667                -6.559149                  1                       2.801700
RISCO_CRIME    alto_modelo_baixo_hurdle         EBM_alto_HurdleNB_baixo    0.023401               -14.934156                  0                      -4.876608

{'__name__': '__main__',
 '__doc__': 'Create empirical evidence tables for why EWMA3 is a strong benchmark.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'evidence_why_ewm3_strong_h3.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __impor

## Como interpretar as evidências

A EWM3 vence a média histórica fixa nos três alvos. Atualizar o nível de cada célula com observações recentes é valioso. A correlação temporal relevante é calculada após subtrair o nível médio de treino de cada H3; correlações brutas misturam persistência com diferenças permanentes entre células.

A divisão de grupos concentra ocorrências, mas não garante previsibilidade adicional. A correção residual testa se as features conseguem antecipar a parte que a EWM3 erra. Os ganhos médios observados nesta rodada são positivos para os três melhores ajustes, porém pequenos para CRIME e especialmente FURTO; só ROUBO venceu em todos os três folds. Com três folds, esses ganhos devem ser tratados como promissores, mas ainda frágeis, e não como prova de superioridade futura.

Correlação e importância preditiva não demonstram causalidade. A validação é rolling one-step-ahead: observações anteriores dentro da janela de validação alimentam o histórico do mês seguinte. Para prever vários meses sem atualizar os registros, será necessária uma validação de origem fixa.